# Comparing Wannierisation minimizers — SGD, Adam, CG, L-BFGS, DIIS, RTR

`core.optim.minimize_spread` minimizes the Marzari-Vanderbilt spread functional $\Omega(U)$ over the unitary gauge $U(k) \in U(n_w)$ using one of six native Riemannian optimizers (no `torch.optim`/`geoopt` — neither supports the complex128 QR retraction this project relies on):

- **SGD** — steepest descent, Armijo backtracking line search.
- **Adam** — Riemannian Adam (Bécigneul & Ganea, ICLR 2019), momentum and per-element scaling carried in the tangent space.
- **CG** — nonlinear conjugate gradients (Fletcher-Reeves), matching wannier90's own minimizer; a parabolic (not Armijo) line search fits the exact quadratic through the current point, its slope, and one trial step.
- **L-BFGS** — quasi-Newton, two-loop recursion over the last few curvature pairs $(s_i, y_i)$ approximating the inverse-Hessian action on the gradient, same parabolic line search as CG.
- **DIIS** — Pulay-mixing (subspace acceleration): each iterate's Riemannian gradient serves as its DIIS "error vector" (the residual of the fixed-point problem $\nabla\Omega=0$); the extrapolated ambient combination of recent iterates is re-projected onto $U(n_w)$ (nearest-unitary/polar retraction) and only accepted if it beats a safeguarded steepest-descent step from the same point.
- **RTR** — Riemannian trust-region (Absil, Baker & Gallivan, *Found. Comput. Math.* **7**, 303 (2007)), the only **second-order** method here: builds a local quadratic model from the Riemannian gradient and an (approximate) Riemannian Hessian-vector product — via double backward through the very same autodiff'd $\Omega(U)$ the gradient already uses, no hand-derived second derivative needed — approximately minimizes it within a trust region via truncated CG (Steihaug-Toint), and accepts/rejects the step while growing/shrinking the trust radius from the ratio of actual to predicted decrease.

All six share the exact same Riemannian-gradient/QR-retraction machinery (`core.optim._riemannian_gradient`/`_qr_retract`) and the same autodiff'd $\Omega(U)$ forward pass — only the search direction and step-size logic differ.

**The hard case**: `w90tutorial/27_silicon_scdm`'s "gaussian" SCDM-entanglement variant — 4 MLWFs built from Si's 4 lowest *conduction* bands (`scdm_entanglement='gaussian', mu=12.5 eV, sigma=2.0 eV`), an exactly-isolated 4-band manifold (no disentanglement freedom at all — every candidate band survives the window at every k, so the whole gauge problem reduces to spread minimization alone). That earlier notebook found this a genuinely slow, asymmetric optimization landscape with more than one local minimum: real `wannier90.x` itself needs ~3000 iterations here too (confirmed from its own `.wout` log), and plain optimizers without `guiding_centres` converge to a different, WORSE local minimum than the one matching wannier90.x's reference value ($\Omega_\mathrm{total}=21.648613\,\mathrm{\mathring{A}}^2$) — exactly the kind of case that can tell optimizers apart.

In [ ]:
import os, sys, pathlib
import numpy as np
import matplotlib.pyplot as plt

HERE = pathlib.Path.cwd()
REPO = HERE
while not (REPO / 'waw').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import waw
from waw.interfaces import quantum_espresso as qe   # the direct-input QE driver
from waw.interfaces.ase.driver import wannierize
from waw.units import BOHR_TO_ANG, HARTREE_TO_EV, EV_TO_HARTREE
from waw.vis import plot_bands, BandSeries

PSEUDO_DIR = REPO / 'workflows' / 'pseudos'   # shared across workflows/, not notebooks/-specific
NCORES = 16
waw.set_num_threads(NCORES)
print('waw', 'threads =', waw.get_num_threads(), '| repo', REPO)

### 1. Structure and DFT — same recipe as tutorial 27's `si_gau`
Bulk diamond Si, $a=5.43$ Å. `outer_window=(6.5, 17.0)` eV excludes the valence manifold entirely (Si's own valence tops out at 6.23 eV, conduction bottoms out at 6.94 eV) — exactly 4 candidate bands survive at every k, so disentanglement is trivial and every optimizer is solving the identical, fixed 4-band gauge problem.

In [ ]:
from ase.build import bulk
from waw.core.optim import minimize_spread

atoms = bulk('Si', 'diamond', a=5.43)
MP_GRID = (4, 4, 4)
WORK = HERE / 'runs' / 'optim_compare'

ov = qe.generate_overlaps(
    atoms, MP_GRID, WORK, 'si_gau',
    ecutwfc=40, scf_kpts=(8, 8, 8), nbnd=8, num_wann=4,
    scdm_entanglement='gaussian', scdm_mu=12.5, scdm_sigma=2.0,
    pseudopotentials={'Si': 'Si.upf'},
    pseudo_dir=PSEUDO_DIR, ncores=NCORES, rerun_scf=False,
)
print('overlaps ready, nk =', len(ov['kpts']))

### 2. Disentangle once, then hand the SAME initial gauge to every optimizer
Disentanglement (subspace selection) is identical regardless of which gauge optimizer runs afterwards, so it's done once here and every optimizer starts from the exact same `U_init` — an apples-to-apples comparison, not five independent Wannierisations that happen to start from re-derived-but-nominally-equal subspaces.

In [ ]:
import torch
from waw.core.disentangle import disentangle
from waw.core.init import svd_init
from waw.core.spread import rotate_overlaps
from waw.interfaces.ase.driver import build_wannier_data
from waw.interfaces.ase.structure import recip_lattice

wdata = build_wannier_data(
    recip_lattice(atoms), ov['kpts'], ov['mmn'], ov['amn'], ov['eig'],
    ov['nnkpts'], ov['g_vectors'],
)
dis = disentangle(wdata.Mmn, wdata.eig, wdata.wb, wdata.kb_idx, nw=4, Amn=wdata.Amn,
                  outer_window=(6.5 * EV_TO_HARTREE, 17.0 * EV_TO_HARTREE))
Mmn_opt = rotate_overlaps(dis.V, wdata.Mmn, wdata.kb_idx)   # (nk, nnb, nw, nw), projected by V
Amn_sub = torch.einsum('kmi,kmj->kij', dis.V.conj(), wdata.Amn)
U_init = svd_init(Amn_sub)
print('U_init built from the shared disentangled subspace:', U_init.shape)

### 3. Run all six optimizers from the same `U_init`, `guiding_centres=True`
`lr` plays a different role per optimizer: a fixed Riemannian-Adam step vs. a line-search trial-step length for the other four. Unlike the easier valence-only Silicon case (`test_cg_is_insensitive_to_trial_step` in `tests/test_init_optim.py`), this harder conduction-manifold landscape turns out NOT to be trial-step-insensitive: CG with `lr=1.0` (a value that works fine on the easy case) lands 6 Ų above wannier90.x's own reference here, while `lr=3e-2` (this notebook's own tutorial-27 recipe never overrides `lr`, i.e. uses this same default) reaches it — a genuine, worth-noting landscape-dependence, not a fixed rule of thumb. All five optimizers use `lr=3e-2` below for a fair, uniform comparison.

In [ ]:
import time

W90_REF = 21.648613   # Ang^2, wannier90.x's own reference for this exact system
OPTIMIZERS = [('sgd', 3e-2), ('adam', 3e-2), ('cg', 3e-2), ('lbfgs', 3e-2),
              ('diis', 3e-2), ('rtr', 0.1)]   # rtr's lr is a trust-region radius, not a step length

results = {}
for name, lr in OPTIMIZERS:
    t0 = time.time()
    res = minimize_spread(
        U_init, Mmn_opt, wdata.wb, wdata.bvecs, wdata.kb_idx,
        optimizer=name, lr=lr, n_iter=3000, conv_tol=1e-10, conv_window=10,
        guiding_centres=True,
    )
    elapsed = time.time() - t0
    results[name] = (res, elapsed)
    omega_ang2 = res.Omega * BOHR_TO_ANG**2
    print(f'{name:6s}: Omega={omega_ang2:9.4f} Ang^2  '
          f'iters={len(res.history):5d}  converged={res.converged!s:5s}  '
          f'time={elapsed:6.2f}s')

### 4. Convergence curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
for name, _ in OPTIMIZERS:
    res, _ = results[name]
    omega_hist = np.array(res.history) * BOHR_TO_ANG**2
    ax.semilogy(np.arange(len(omega_hist)), omega_hist - W90_REF + 1e-6, label=name)
ax.axhline(1e-6, color='0.6', lw=0.7, ls=':')
ax.set_xlabel('iteration'); ax.set_ylabel(r'$\Omega - \Omega_{W90}$ (Ang$^2$)')
ax.set_title('Silicon conduction-manifold SCDM — optimizer convergence')
ax.legend(); fig.tight_layout()

### 5. Summary table

In [ ]:
print(f'{"optimizer":8s} {"Omega (Ang^2)":>14s} {"iters":>7s} '
      f'{"converged":>10s} {"time (s)":>9s} {"vs W90":>10s}')
for name, _ in OPTIMIZERS:
    res, elapsed = results[name]
    omega_ang2 = res.Omega * BOHR_TO_ANG**2
    print(f'{name:8s} {omega_ang2:14.6f} {len(res.history):7d} '
          f'{str(res.converged):>10s} {elapsed:9.2f} {omega_ang2 - W90_REF:+10.4f}')

### 6. Without `guiding_centres`
Tutorial 27's original notebook found that plain optimizers can land in a different, worse local minimum on this system without `guiding_centres`. Re-running without it, starting from the exact same shared `U_init` used above, isolates whether that's a `guiding_centres` effect specifically, or something else about this landscape.

In [ ]:
results_noguide = {}
for name, lr in OPTIMIZERS:
    res = minimize_spread(
        U_init, Mmn_opt, wdata.wb, wdata.bvecs, wdata.kb_idx,
        optimizer=name, lr=lr, n_iter=3000, conv_tol=1e-10, conv_window=10,
        guiding_centres=False,
    )
    results_noguide[name] = res
    omega_ang2 = res.Omega * BOHR_TO_ANG**2
    print(f'{name:6s} (no guiding_centres): Omega={omega_ang2:9.4f} Ang^2  '
          f'(with guiding_centres: {results[name][0].Omega * BOHR_TO_ANG**2:.4f})')

**Takeaway.** On this particular comparison (identical shared `U_init`, uniform `lr`), `guiding_centres` turns out to change very little — every optimizer's final Omega moves by well under 0.2 Ų either way. The real fault line is between optimizer FAMILIES: **CG and Adam** reach wannier90.x's reference ($21.648613\,\mathrm{\mathring{A}}^2$) to within $0.1$-$0.4\,\mathrm{\mathring{A}}^2$, while **SGD, L-BFGS, DIIS, and RTR** all settle into a distinctly different, shallower local minimum around $28.3$-$28.7\,\mathrm{\mathring{A}}^2$ instead. That's not a coincidence for the first three: SGD's step *is* Armijo-backtracked steepest descent, and this notebook's L-BFGS/DIIS both fall back to exactly that same Armijo steepest-descent step whenever their own candidate step fails to lower Omega (L-BFGS: an ill-scaled quasi-Newton direction; DIIS: a Pulay-extrapolated point that doesn't improve on it) — so all three inherit steepest descent's basin of attraction on this landscape. RTR's presence in the same basin is the more surprising result: it never falls back to steepest descent at all (a rejected trust-region step just shrinks the radius and retries from the same point) and uses genuine second-order curvature information, yet it still converges — extremely fast (~110 iterations, a couple of seconds, using real Hessian-vector products) — to the *same* shallower minimum SGD/L-BFGS/DIIS find, not CG/Adam's deeper one. This is a general feature of nonconvex optimization landscapes: a locally accurate, fast-converging optimizer (RTR, and L-BFGS once its steps are safeguarded) reaches whichever local minimum's basin of attraction it started in *efficiently* — it has no special ability to escape a bad basin just because it's a better local method. Only CG's periodic steepest-descent restarts and parabolic line search, and Adam's momentum, happen to wander far enough off the straight-downhill path from this particular `U_init` to fall into the deeper basin instead. Which basin a run lands in here is decided by the optimizer's *trajectory*, not by how good it is at locally minimizing once it's near a minimum.